# Modelado, Intervenciones de Fairness y Explicabilidad (XAI)

Este notebook integra los módulos de preprocesamiento y entrenamiento para entrenar un modelo Baseline, evaluar el sesgo y aplicar intervenciones de equidad algorítmica utilizando `fairlearn`.

In [ ]:
import pandas as pd
import os
import sys
import shap
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from features.build_features import get_data_splits, build_preprocessor
from models.train_model import train_baseline, train_fair_model
from models.evaluate import evaluate_model, get_fairness_summary

shap.initjs()

## 1. Carga de Datos y Preprocesamiento
Tal como se especifica en la guía, evitamos el **Data Leakage** dividiendo los datos ANTES de cualquier transformación (e.g., One-Hot Encoding).

In [ ]:
data_path = os.path.join('..', 'data', 'processed', 'compas_cleaned.csv')
df = pd.read_csv(data_path)

# Features numéricas y categóricas
numeric_features = ['age', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count']
categorical_features = ['sex', 'c_charge_degree']

X_train, X_test, y_train, y_test, A_train, A_test = get_data_splits(
    df, 
    target_col='two_year_recid',
    sensitive_col='race',
    test_size=0.2,
    random_state=42
)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

preprocessor = build_preprocessor(numeric_features, categorical_features)

## 2. Entrenamiento Baseline (Sin mitigación)

In [ ]:
baseline_model = train_baseline(X_train, y_train, preprocessor, random_state=42)

y_pred_baseline = baseline_model.predict(X_test)

print("Evaluación Baseline (Logistic Regression):")
df_metrics_baseline = evaluate_model(y_test, y_pred_baseline, A_test)
display(df_metrics_baseline)

Podemos observar que en el baseline, el **False Positive Rate (FPR)** suele ser mayor para afroamericanos y el **False Negative Rate (FNR)** mayor para caucásicos, similar a lo reportado por ProPublica.

## 3. Mitigación de Sesgo (Post-processing)
Aplicaremos `ThresholdOptimizer` de Fairlearn para igualar las tasas de error (*Equalized Odds*).

In [ ]:
# Entrenamos el modelo justo (ajusta umbrales de decisión basado en el grupo racial)
fair_model = train_fair_model(
    baseline_model, 
    X_train, 
    y_train, 
    A_train, 
    constraint="equalized_odds", 
    random_state=42
)

y_pred_fair = fair_model.predict(X_test, sensitive_features=A_test)

print("Evaluación Modelo Mitigado (Equalized Odds):")
df_metrics_fair = evaluate_model(y_test, y_pred_fair, A_test)
display(df_metrics_fair)

Comparamos ambos:

In [ ]:
print("Diferencia de paridad demográfica y probabilidades igualadas")
print("Baseline:", get_fairness_summary(y_test, y_pred_baseline, A_test))
print("Fair Model:", get_fairness_summary(y_test, y_pred_fair, A_test))

## 4. Explicabilidad con SHAP
Dado que usamos un pipeline con un preprocesador y un regresor logístico, extraeremos los coeficientes o usaremos LinearExplainer.

In [ ]:
# Para SHAP necesitamos transformar los datos primero para dárselos al estimador final
X_train_transformed = baseline_model.named_steps['preprocessor'].transform(X_train)
X_test_transformed = baseline_model.named_steps['preprocessor'].transform(X_test)

# Nombres de las features
numeric_feature_names = numeric_features
categorical_feature_names = baseline_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
feature_names = list(numeric_feature_names) + list(categorical_feature_names)

explainer = shap.LinearExplainer(baseline_model.named_steps['classifier'], X_train_transformed)
shap_values = explainer.shap_values(X_test_transformed)

shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names)